# ⭐ Actividad 07: Diseño del Star Schema — Data Warehouse
---
**Objetivo:** Documentar visualmente el modelo dimensional (Star Schema) para `limon_analytics_db`.

Este es el diseño que soportará las consultas analíticas del modelo LSTM-Attention, permitiendo
cruzar producción agrícola con emergencias, noticias y datos climáticos.

---

## 7.1 Modelo Estrella — Diagrama Visual

```
                    ┌─────────────────────────────┐
                    │        dim_tiempo            │
                    │─────────────────────────────│
                    │ id_tiempo (PK)              │
                    │ fecha_evento    VARCHAR(7)   │
                    │ anho            SMALLINT     │
                    │ mes             SMALLINT     │
                    │ trimestre       SMALLINT     │
                    │ month_sin       FLOAT        │
                    │ month_cos       FLOAT        │
                    └──────────┬──────────────────┘
                               │
                               │ 1:N
                               │
┌─────────────────────────────┐│┌────────────────────────────────────────────┐
│      dim_ubicacion          │││         fact_produccion_limon               │
│─────────────────────────────│││────────────────────────────────────────────│
│ id_ubicacion (PK)           │││ id_hecho (PK)                             │
│ departamento  VARCHAR(60)   ├┤│ id_tiempo (FK) ─────────────► dim_tiempo  │
│ provincia     VARCHAR(60)   │││ id_ubicacion (FK) ──────────► dim_ubicacion│
│ lat           FLOAT         │││                                            │
│ lon           FLOAT         │││ ── MÉTRICAS AGRÍCOLAS (MIDAGRI) ──        │
└─────────────────────────────┘││ produccion_t         FLOAT                │
                               ││ cosecha_ha           FLOAT                │
                               ││ precio_chacra_kg     FLOAT                │
                               ││                                            │
```

## 7.1 Modelo Estrella — Diagrama Multimodal (4 Dimensiones)

Este diseño separa las fuentes de datos en 4 dimensiones clave para permitir un análisis granular del impacto climático y social en la producción de limón.

### Vista Conceptual (Esquema de la Tesis)
![Star Schema Diagram](../data/04_reports/g07_star_schema_v2.png)

---

### Vista Técnica (Modelo Dimensional)
```mermaid
erDiagram
    fact_produccion_limon }|--|| dim_tiempo : "FK_Tiempo"
    fact_produccion_limon }|--|| dim_ubicacion : "FK_Ubicacion"
    fact_produccion_limon }|--|| dim_clima : "FK_Clima"
    fact_produccion_limon }|--|| dim_multimodal : "FK_Multimodal"

    dim_tiempo {
        int id_tiempo PK
        varchar fecha_evento "YYYY-MM"
        int anho
        int mes
        float month_sin "Estacionalidad"
        float month_cos "Estacionalidad"
    }

    dim_ubicacion {
        int id_ubicacion PK
        varchar departamento "Región"
        varchar provincia
        float lat
        float lon
    }

    dim_clima {
        int id_clima PK
        float temp_max_c "NASA"
        float temp_min_c "NASA"
        float precipitacion_mm "NASA"
        float radiacion_solar "NASA"
    }

    dim_multimodal {
        int id_multimodal PK
        int n_noticias "NLP"
        int num_emergencias "INDECI"
        float avg_sentimiento "NLP"
        int total_afectados "INDECI"
    }

    fact_produccion_limon {
        int id_hecho PK
        int id_tiempo FK
        int id_ubicacion FK
        int id_clima FK
        int id_multimodal FK
        float produccion_t "Métrica Principal"
        float cosecha_ha
        float precio_chacra_kg "Volatilidad"
    }
```

---

## 7.2 Justificación del Diseño

| Decisión | Justificación |
|:---------|:-------------|
| **Star Schema** | Optimiza consultas analíticas con JOINs simples. Ideal para alimentar el modelo LSTM-Attention. |
| **Granularidad mensual** | La producción agrícola se reporta mensualmente. NASA e INDECI se agregan a este nivel. |
| **Dimensión Clima** | Independiza las variables de la NASA para permitir análisis histórico de correlación. |
| **Dimensión Multimodal** | Consolida el riesgo social (Noticias) y desastres (INDECI) en una sola jerarquía de impacto. |

---

## 7.3 Relaciones y Cardinalidad

| Relación | Tipo | Descripción |
|:---------|:-----|:------------|
| `fact → dim_tiempo` | N:1 | Muchos registros de producción pueden compartir el mismo mes |
| `fact → dim_ubicacion` | N:1 | Muchos registros de producción pueden compartir la misma provincia |
| `fact → dim_clima` | 1:1 | Cada registro de producción tiene un perfil climático único |
| `fact → dim_multimodal` | 1:1 | Cada registro tiene un contexto de noticias y emergencias asociado |
